In [0]:
storage_account_name = "flightstacc"
storage_account_key = dbutils.secrets.get(scope = 'wikimedia', key = 'Access-key')

spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)
display(dbutils.fs.ls(f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/"))

In [0]:

stream_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/opensky/wikipedia-live-stream"

df_avro = (spark.read.format("avro")
           .option("recursiveFileLookup", "true")
           .option("ignoreExtension", "true")
           .load(stream_path))

display(df_avro.limit(10))

In [0]:
from pyspark.sql.functions import col, from_json, schema_of_json
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

df_decoded = df_avro.withColumn("body_str", col("Body").cast("string"))
sample_json = df_decoded.select("body_str").first()["body_str"]
json_schema = schema_of_json(sample_json)

df_cleaned = df_decoded.withColumn("data", from_json(col("body_str"), json_schema)) \
                       .withColumn("sequence_number", col("SequenceNumber")) \
                       .withColumn("enqueued_time", col("EnqueuedTimeUtc")) \
                       .select("sequence_number", "enqueued_time", "data.*")

display(df_cleaned)

In [0]:
df_final_stream = df_cleaned.select(
    "sequence_number",
    "enqueued_time",
    "id",
    "timestamp",
    "title",
    "bot",        
    "user",
    "type",
    "wiki"
)

display(df_final_stream)

In [0]:
# 1. Stream Data write to Parquet
output_stream_path = "abfss://bronze@flightstacc.dfs.core.windows.net/snowflake_staging_api/stream_data/"
df_final_stream.write.mode("overwrite").parquet(output_stream_path)
print("Data written successfully to ADLS Gen2!")